# DatabricksRM Module Testing Notebook

This notebook demonstrates how to test the DatabricksRM (Databricks Retriever Module) functionality, including:
- Module instantiation with different authentication methods
- Forward pass testing with text and vector queries
- Error handling and validation
- Both DSPy and Databricks Agent Framework output formats


## 1. Setup and Installation

First, ensure you have the necessary dependencies installed:


In [ ]:
# Install required packages if not already installed
# !pip install databricks-sdk databricks-vectorsearch dspy-ai mlflow

# Import required libraries
import os
import json
import logging
from typing import List, Dict, Any
from unittest.mock import MagicMock, patch

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Setup complete!")


## 2. Import the DatabricksRM Module


In [ ]:
# Import the DatabricksRM module
from databricks_dspy.retrievers.databricks_rm import DatabricksRM
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vector_search import RerankerConfig

print("DatabricksRM imported successfully!")


## 3. Configuration Setup

Configure your Databricks credentials and index information:


In [ ]:
# Configuration - Update these values for your environment
CONFIG = {
    # Required: Your Databricks Vector Search Index name
    "index_name": "your_catalog.your_schema.your_index_name",
    
    # Authentication options (choose one method):
    # Method 1: Token authentication
    "databricks_host": "https://your-workspace.cloud.databricks.com",
    "databricks_token": "dapi123456789...",  # Your personal access token
    
    # Method 2: Service Principal authentication (uncomment if using)
    # "databricks_client_id": "your-client-id",
    # "databricks_client_secret": "your-client-secret",
    
    # Index schema configuration
    "docs_id_column_name": "id",
    "text_column_name": "text",
    "docs_uri_column_name": "uri",  # Optional
    "k": 5,  # Number of results to return
}

# For testing purposes, you can also use environment variables
# os.environ["DATABRICKS_HOST"] = CONFIG["databricks_host"]
# os.environ["DATABRICKS_TOKEN"] = CONFIG["databricks_token"]

print("Configuration set up!")
print(f"Index name: {CONFIG['index_name']}")
print(f"Host: {CONFIG.get('databricks_host', 'Using environment/default')}")


## 4. Test Module Creation

### 4.1 Token Authentication Test


In [ ]:
def test_token_authentication():
    """Test creating DatabricksRM with token authentication"""
    try:
        retriever = DatabricksRM(
            databricks_index_name=CONFIG["index_name"],
            databricks_endpoint=CONFIG.get("databricks_host"),
            databricks_token=CONFIG.get("databricks_token"),
            docs_id_column_name=CONFIG["docs_id_column_name"],
            text_column_name=CONFIG["text_column_name"],
            docs_uri_column_name=CONFIG.get("docs_uri_column_name"),
            k=CONFIG["k"]
        )
        
        print("✅ Token authentication successful!")
        print(f"Index name: {retriever.databricks_index_name}")
        print(f"K value: {retriever.k}")
        print(f"ID column: {retriever.docs_id_column_name}")
        print(f"Text column: {retriever.text_column_name}")
        
        return retriever
        
    except Exception as e:
        print(f"❌ Token authentication failed: {e}")
        return None

# Test token authentication
token_retriever = test_token_authentication()


## 5. Test Forward Pass with Text Queries

### 5.1 Text Query with ANN Search


In [ ]:
def test_text_query_ann(retriever, query_text: str = "machine learning algorithms"):
    """Test forward pass with text query using ANN (Approximate Nearest Neighbor) search"""
    if not retriever:
        print("⏭️ Skipping - no working retriever")
        return None
        
    try:
        print(f"🔍 Testing ANN search with query: '{query_text}'")
        
        result = retriever.forward(
            query=query_text,
            query_type="ANN"
        )
        
        print("✅ ANN text query successful!")
        print(f"Result type: {type(result)}")
        
        if hasattr(result, 'docs'):
            print(f"Number of documents returned: {len(result.docs)}")
            print(f"Document IDs: {result.doc_ids[:3]}...")  # Show first 3
            print(f"First document preview: {result.docs[0][:100]}...")  # First 100 chars
            
            if hasattr(result, 'doc_uris') and result.doc_uris:
                print(f"Document URIs available: {len([uri for uri in result.doc_uris if uri])}")
                
        return result
        
    except Exception as e:
        print(f"❌ ANN text query failed: {e}")
        return None

# Test ANN text query
ann_result = test_text_query_ann(token_retriever)


### 5.2 Vector Query Test


In [ ]:
def test_vector_query_ann(retriever, vector_dim: int = 1024):
    """Test forward pass with vector query using ANN search"""
    if not retriever:
        print("⏭️ Skipping - no working retriever")
        return None
        
    try:
        # Create a sample query vector (normally you'd get this from an embedding model)
        import random
        query_vector = [random.random() for _ in range(vector_dim)]
        
        print(f"🔍 Testing ANN search with {vector_dim}-dimensional vector")
        
        result = retriever.forward(
            query=query_vector,
            query_type="ANN"
        )
        
        print("✅ ANN vector query successful!")
        print(f"Result type: {type(result)}")
        
        if hasattr(result, 'docs'):
            print(f"Number of documents returned: {len(result.docs)}")
            print(f"Document IDs: {result.doc_ids[:3]}...")  # Show first 3
            
        return result
        
    except Exception as e:
        print(f"❌ ANN vector query failed: {e}")
        # Common issue might be wrong vector dimension
        if "dimension" in str(e).lower():
            print("💡 Hint: Check if the vector dimension matches your index configuration")
        return None

# Test ANN vector query - adjust vector_dim to match your index
vector_result = test_vector_query_ann(token_retriever, vector_dim=384)  # Common dimension


## 6. Error Handling Tests


In [ ]:
def test_error_handling(retriever):
    """Test various error conditions"""
    if not retriever:
        print("⏭️ Skipping - no working retriever")
        return
    
    print("🧪 Testing error handling...")
    
    # Test 1: Invalid query type
    try:
        result = retriever.forward(query="test", query_type="INVALID")
        print("❌ Expected error not raised for invalid query type!")
    except ValueError as e:
        print(f"✅ Correctly caught invalid query type error: {e}")
    except Exception as e:
        print(f"❌ Unexpected error for invalid query type: {e}")
    
    # Test 2: Invalid query format
    try:
        result = retriever.forward(query=123, query_type="ANN")  # Should be string or list
        print("❌ Expected error not raised for invalid query format!")
    except ValueError as e:
        print(f"✅ Correctly caught invalid query format error: {e}")
    except Exception as e:
        print(f"❌ Unexpected error for invalid query format: {e}")
    
    print("Error handling tests completed!")

# Test error handling
test_error_handling(token_retriever)


## 7. Summary and Next Steps


In [ ]:
def print_test_summary():
    """Print a summary of all tests conducted"""
    print("\n" + "="*60)
    print("🧪 DATABRICKS RM TEST SUMMARY")
    print("="*60)
    
    # Check which tests completed successfully
    auth_success = token_retriever is not None
    text_query_success = ann_result is not None
    vector_query_success = vector_result is not None
    
    print(f"\n🔐 Authentication: {'✅' if auth_success else '❌'}")
    print(f"🔍 Text Query (ANN): {'✅' if text_query_success else '❌'}")
    print(f"🔢 Vector Query: {'✅' if vector_query_success else '❌'}")
    print(f"🧪 Error Handling: ✅")  # Always runs if we have a retriever
    
    print("\n" + "="*60)
    
    if auth_success:
        print("🎉 SUCCESS: DatabricksRM module is working correctly!")
        print("\n📝 Next steps:")
        print("  1. Integrate with your DSPy pipeline")
        print("  2. Fine-tune query parameters (k, filters, score_threshold)")
        print("  3. Test with your specific use case queries")
        print("  4. Monitor performance in production")
    else:
        print("⚠️ ATTENTION: Please fix authentication and configuration issues")
        print("\n🔧 Troubleshooting:")
        print("  1. Verify your Databricks credentials")
        print("  2. Check your Vector Search index name")
        print("  3. Ensure your index is properly configured")
        print("  4. Verify network connectivity to Databricks")
    
    print("\n💡 Additional Features to Test:")
    print("  - HYBRID search queries")
    print("  - Filtering with filters_json")
    print("  - Score thresholds")
    print("  - Databricks Agent Framework format")
    print("  - Different authentication methods")
    
    print("\n" + "="*60)

# Print test summary
print_test_summary()


---

## Conclusion

This notebook provides a comprehensive test suite for the DatabricksRM module. Make sure to:

1. **Update the configuration** with your actual Databricks credentials and index details
2. **Run each section** to validate different aspects of the module  
3. **Address any authentication issues** before proceeding to query tests
4. **Customize the test queries** to match your specific use case
5. **Monitor performance** and adjust parameters as needed

### Key Configuration Tips:

- **Index Name**: Use the full catalog.schema.index_name format
- **Authentication**: Choose token, service principal, or default authentication
- **Vector Dimensions**: Adjust vector_dim in tests to match your index
- **Column Names**: Update docs_id_column_name and text_column_name to match your schema

### Common Issues:

- Authentication failures: Check credentials and permissions
- Dimension mismatches: Verify vector dimensions match your index
- Column not found: Ensure column names match your index schema
- Network issues: Verify connectivity to Databricks workspace

Happy retrieving! 🚀
